# nb01 — Build the ReliefForge datasets (Kaggle, CPU)

Run with **Settings → Accelerator: none (CPU), Internet: on**.

1. **PRIMARY (real):** download `KarimT/Relief-Depth` (~11 GB) and convert it to our
   height convention (`h = |depth − slab|`, splits respected). 807 scenes × 5 lightings.
2. **AUXILIARY (synthetic):** generate 1,500 procedural scenes (exact labels).

After Save & Run All: attach `/kaggle/working/dataset` (synthetic) and
`/kaggle/working/relief_depth` (real) as Kaggle Datasets for nb02.

In [ ]:
!nvidia-smi 2>/dev/null || echo "no GPU (this notebook is CPU-only - that is fine)"
!nproc && free -g | head -2 && df -h /kaggle/working | tail -1

In [ ]:
%%bash
cd /kaggle/working
rm -rf reliefforge
git clone --depth 1 https://github.com/<YOUR_USERNAME>/reliefforge.git reliefforge
cd reliefforge
pip -q install PyYAML

In [ ]:
%%bash
cd /kaggle/working/reliefforge
# PRIMARY: real Relief-Depth pairs (download ~11 GB + convert)
python src/data_download.py --out /kaggle/working/dataset --n-scenes 1500 --seed0 0 --workers 4 \n  --relief-depth --real-out /kaggle/working/relief_depth --raw-dir /kaggle/working/reliefdepth_raw

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/reliefforge/src")
from synth.generate import contact_sheet

sheet = contact_sheet("/kaggle/working/dataset", n=24)
print("contact sheet:", sheet)
# LOOK at /kaggle/working/dataset/contact_sheet.png before training (QC gate, PRD_v2 §5.7)

In [ ]:
!du -sh /kaggle/working/dataset && ls /kaggle/working/dataset
!cat /kaggle/working/dataset/scenes.jsonl | head -2

**After Save & Run All finishes:** Output tab → New Dataset (`reliefforge-dataset`),
then in `nb02_train` add it as **Add Input → Your Work → reliefforge-dataset**
(it will appear at `/kaggle/input/reliefforge-dataset`).

Optional (recommended for the course form): publish the dataset folder to the
Hugging Face Hub with `huggingface_hub` (HF token as a Kaggle Secret `HF_TOKEN`).